# RAG 전처리 파이프라인
**HWP / PDF 원문 → 구조화 JSON 변환**

---

## 전체 흐름

```
원본 파일 100개 (HWP 96 + PDF 4)
      +
메타데이터 CSV (data_list.csv)
      │
      ▼
[Phase 0] 환경 설정 & 라이브러리 설치
[Phase 1] 드라이브 마운트 & 경로 설정
[Phase 2] 사전 제거 & CSV 전처리 (결측치 <unknown>, 파일 매핑)
[Phase 3] HWP 파싱 유틸리티 정의
[Phase 4] PDF 파싱 유틸리티 정의
[Phase 5] JSON 빌드 함수 정의
[Phase 6] 파싱 예시 출력 (HWP 1건 + PDF 1건)
[Phase 7] 전체 100건 파이프라인 실행
[Phase 8] manifest 저장 & 재공고 연결
[QA]      샘플 검증
      │
      ▼
내드라이브 > part3data > Preprocessed dataset
  ├─ docs/D001.json ~ D100.json
  ├─ manifest.csv
  └─ failed_docs.csv (실패 시 생성)
```

---

## JSON 산출물 스키마 요약

| 필드 | 설명 |
|---|---|
| `metadata` | CSV에서 가져온 사업 정보 (결측치는 `<unknown>`) |
| `toc` | 원문에서 추출한 목차 항목 리스트 |
| `sections[]` | 헤더 기준으로 분할된 섹션 배열 |
| `sections[].header_path` | 해당 섹션의 목차 계층 경로 (예: `["Ⅰ. 사업개요", "1. 사업범위"]`) |
| `sections[].blocks[]` | 섹션 내 본문/표 블록 배열 |
| `blocks[].type` | `"text"` 또는 `"table"` |
| `blocks[].content` | **청킹에서 바로 사용할 텍스트** |
| `blocks[].table_type` | `"wide"` (Markdown 표) 또는 `"key_value"` (키:값 문장) |
| `blocks[].raw_grid` | 표의 2D 원본 배열 (재가공·QA 용도) |
| `qa` | 전처리 품질 정보 (경고, 섹션/블록 수 등) |

---
##0.환경 설정 & 라이브러리 설치

### 선택 근거
| 라이브러리 | 용도 | 선택 이유 |
|---|---|---|
| `pyhwp 0.1b15` | HWP 파싱 | HWP 내부 OLE 스트림을 레코드 레벨까지 파싱. 단순 텍스트 추출(`hwp5txt`)이 아닌 **표의 row/col/span 메타를 보존**하는 유일한 Python 라이브러리 |
| `pdfplumber 0.11.9` | PDF 파싱 | pdfminer 기반으로 텍스트 추출 + 표 경계 감지를 함께 수행. 디지털 생성 PDF에서 셀 좌표까지 복원 가능 |
| `lxml 6.0.2` | HWP XML 파싱 | pyhwp가 생성한 XML을 ElementTree보다 빠르게 탐색. `iterancestors()` 등 고급 탐색 필요 |
| `pandas 2.2.2` | CSV 처리 | 결측치 처리·필터링·join에 최적화 |

In [ ]:
!pip install pyhwp==0.1b15 pdfplumber==0.11.9 lxml==6.0.2 'pandas==2.2.2' -q
print('설치 완료')

In [ ]:
import os, re, json, hashlib, subprocess, tempfile, unicodedata
from pathlib import Path
from datetime import date
from difflib import SequenceMatcher  # 표/텍스트 유사도 비교용 (decorative table 판별에 사용)

import pandas as pd
import pdfplumber
from lxml import etree

print('라이브러리 로드 완료')


---
##1.드라이브 마운트 & 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BASE_DIR   = Path('/content/drive/MyDrive/part3data')
FILES_DIR  = BASE_DIR / 'files'
CSV_PATH   = BASE_DIR / 'data_list.csv'
OUTPUT_DIR = BASE_DIR / 'Preprocessed dataset'
DOCS_DIR   = OUTPUT_DIR / 'docs'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

all_files = list(FILES_DIR.iterdir())
hwp_files = [f for f in all_files if f.suffix.lower() == '.hwp']
pdf_files = [f for f in all_files if f.suffix.lower() == '.pdf']

print(f'HWP: {len(hwp_files)}건  PDF: {len(pdf_files)}건  전체: {len(all_files)}건')
print(f'출력 경로: {OUTPUT_DIR}')

---
##2.사전 제거 & CSV 전처리

### 2-A. 사전 제거 — 중복 파일

**이유**: `한국한의학연구원_통합정보시스템 고도화 용역` 은 `국가과학기술지식정보서비스_통합정보시스템 고도화 용역` 과  
사업명이 동일하고 실질적으로 동일한 내용의 파일로 확인됨. 중복 데이터가 RAG 학습에 포함될 경우  
동일 내용에 대한 검색 편향이 생길 수 있으므로 CSV 단계에서 선제 제거.

### 2-B. 결측치 처리

####`1.텍스트 <unknown> 치환`

**이유**: `None` 또는 `NaN` 을 그대로 JSON에 넣으면 청킹시 코드에서 타입 오류 발생 가능.  
`<unknown>` 문자열로 통일하면 필터 조건(`metadata.입찰참여시작일 != '<unknown>'`)으로 명시적 처리 가능.  
임의 보간(평균값, 날짜 추정 등)은 하지 않는다 — 없는 정보를 만들어내는 것은 RAG 품질을 해칠 수 있음.

####`2.사업금액 → None `
이유 : "금액 정보 없음"을 의미하며, 실제 0원과 구분하고 통계 왜곡을 방지하기 위해 처리.

####`3.차수 → 0 `
이유 : 실제로 0차 사업은 거의 없으므로 "차수 정보 없음"을 나타내는 코드값으로 사용.


### 2-C. 파일 매핑 — 유니코드 정규화(NFC) 후 정확 일치 매핑

**실제 원인**: CSV `파일명`과 `files/` 디렉토리 파일명은 화면상 동일하게 보이지만,  
유니코드 인코딩 방식이 서로 다름.
- 구글 드라이브(특히 macOS 동기화 경로)는 한글 파일명을 **NFD(분해형)**로 저장하는 경우가 많음  
  예: `정` = `ㅈ`+`ㅓ`+`ㅇ` 코드포인트 3개로 분해
- CSV는 보통 **NFC(조합형)**로 저장됨 — `정` 한 글자가 코드포인트 1개

두 표현은 `==` 비교나 딕셔너리 키 조회에서 **다른 문자열로 취급**되어 100건 전부 매핑 실패가 발생함.

#### **해결**: 비교 전 양쪽 파일명을 `unicodedata.normalize('NFC', ...)`로 통일.

In [ ]:
# ── 2-A. CSV 로드 & 사전 제거 ─────────────────────────────────
df = pd.read_csv(CSV_PATH, encoding='utf-8-sig')
df.columns = [
    '공고번호','공고차수','사업명','사업금액','발주기관',
    '공개일자','입찰참여시작일','입찰참여마감일',
    '사업요약','파일형식','파일명','텍스트'
]
print(f'원본 행 수: {len(df)}')

# 한국한의학연구원 행 제거 (이름만 다른 중복 파일)
drop_mask = df['파일명'].str.contains('한국한의학연구원', na=False)
print(f'제거 대상: {drop_mask.sum()}건')
print(df[drop_mask][['사업명','발주기관','파일명']].to_string())
df = df[~drop_mask].reset_index(drop=True)
print(f'제거 후 행 수: {len(df)}')

총 99개의 문서로 진행 예정

**결과**

원본 행 수: 100


제거 대상: 1건


||사업명|발주기관|                                     파일명|
|---|---|---|---|
|53  |통합정보시스템 |고도화 용역 | 한국한의학연구원  한국한의학연구원_통합정보시스템 고도화 용역.hwp|
제거 후 행 수: 99

In [ ]:
# ── 2-B. 중복 제거 (본문 해시 기준) ──────────────────────────
df['_hash'] = df['텍스트'].apply(lambda x: hashlib.md5(str(x).encode()).hexdigest())
before = len(df)
df = df.drop_duplicates(subset='_hash', keep='first').reset_index(drop=True)
print(f'해시 중복 제거: {before - len(df)}건 → 남은 행: {len(df)}')

# ── doc_id 부여 ───────────────────────────────────────────────
df['doc_id'] = [f'D{str(i+1).zfill(3)}' for i in range(len(df))]

# ── 2-C. 결측치 전체 → <unknown> 치환 ────────────────────────
# 수치형은 별도 처리 (사업금액은 float 유지, 결측만 <unknown>)
print('\n[결측치 현황 - 치환 전]')
print(df.isnull().sum()[df.isnull().sum() > 0])

UNKNOWN = '<unknown>'

# 문자열 컬럼: 결측 → <unknown>
str_cols = ['공고번호','공개일자','입찰참여시작일','입찰참여마감일','사업요약']
for col in str_cols:
    df[col] = df[col].fillna(UNKNOWN).astype(str)

# 공고차수: 결측 → 0
df['공고차수'] = df['공고차수'].fillna(0).astype(int)

# 사업금액: float 유지, 결측 → None (JSON null로 표현)
df['사업금액'] = df['사업금액'].where(df['사업금액'].notna(), other=None)

print('\n[결측치 현황 - 치환 후]')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('(사업금액만 None 허용, 나머지는 <unknown> 처리 완료)')

# ── 사업유형 파생 ─────────────────────────────────────────────
TYPE_KW = [('재구축','재구축'),('고도화','고도화'),('개선','개선'),
           ('개발','개발'),('운영','운영'),('통합','통합'),('구축','구축')]
def infer_type(name):
    for kw, lb in TYPE_KW:
        if kw in str(name): return lb
    return '기타'
df['사업유형'] = df['사업명'].apply(infer_type)

# ── 재공고 여부 ───────────────────────────────────────────────
df['재공고여부'] = df['공고차수'] > 0

print('\nCSV 전처리 완료')
df[['doc_id','사업명','사업유형','재공고여부','파일형식']].head()

파일명 양끝 공백을 제거해 NFC 형식으로 통일 후 파일명 기준 매핑 진행

**결과**

해시 중복 제거: 0건 → 남은 행: 99

[결측치 현황 - 치환 전]

공고번호       18

공고차수       18

사업금액        1

입찰참여시작일    26

입찰참여마감일     8

dtype: int64

[결측치 현황 - 치환 후]

사업금액    1

dtype: int64

(사업금액만 None 허용, 나머지는 <unknown> 처리 완료)

CSV 전처리 완료

||doc_id|	사업명	|사업유형|	재공고여부|	파일형식|
|---|---|---|---|---|---|
|0	|D001|	한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화|	고도화	|False	|hwp
|1|	D002	|2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선|	개선	|False|	hwp
|2	|D003	|EIP3.0 고압가스 안전관리 시스템 구축 용역|	구축	|False|	hwp
|3|	D004	|도시계획위원회 통합관리시스템 구축용역	|통합	|False|	hwp|
|4|	D005	|봉화군 재난통합관리시스템 고도화 사업(협상)(긴급)	|고도화	|False	|hwp


In [ ]:
# ── 2-D. 파일 매핑 (NFC 정규화 후 정확 일치) ─────────────────
#
# 화면상 동일해 보이는 한글 파일명이 NFC/NFD 인코딩 차이로 다른 문자열로
# 취급되는 문제를 해결하기 위해, 비교 전 양쪽 모두 NFC로 정규화한다.

def nfc(s: str) -> str:
    """유니코드 NFC(조합형) 정규화 + 양끝 공백 제거"""
    return unicodedata.normalize('NFC', str(s).strip())

# 실제 파일명 → NFC 정규화 후 딕셔너리 구성
file_map = {nfc(f.name): f for f in all_files}

def find_file(csv_filename: str):
    """CSV 파일명 → 실제 파일 경로 (NFC 정규화 후 정확 일치)"""
    if pd.isna(csv_filename):
        return None
    return file_map.get(nfc(csv_filename))

df['file_path'] = df['파일명'].apply(find_file)

# 매핑 결과 리포트
success = df[df['file_path'].notna()]
failed  = df[df['file_path'].isna()]
print(f'매핑 성공: {len(success)}건 / 실패: {len(failed)}건')

if len(failed) > 0:
    print('\n[매핑 실패 목록] → files/ 디렉토리 파일명 직접 확인 필요')
    print(failed[['doc_id','파일명','파일형식']].to_string(index=False))
else:
    print('전체 매핑 성공 ✅')

**결과**

매핑 성공: 99건 / 실패: 0건

전체 매핑 성공 ✅

---
##3.HWP 파싱 유틸리티

### 처리 흐름
```
HWP 파일
  └─▶ hwp5proc xml (pyhwp CLI)  → XML 스트림
        └─▶ lxml ElementTree 파싱
              ├─▶ TableBody(표) 요소  → 그리드 복원 → 표 직렬화
              └─▶ Paragraph(문단) 요소  → 헤더 감지 or 본문 텍스트
```

### 핵심 설계 결정

**① 표 그리드 복원**  
HWP의 `TableCell` 요소에는 `row`, `col`, `rowspan`, `colspan` 속성이 그대로 들어있음.  
이를 활용해 병합 셀을 포함한 2D 그리드를 정확히 복원.

단순 텍스트 추출(`hwp5txt`)은 이 메타를 버리기 때문에 표 내 셀 경계가 사라짐 → 사용하지 않음.

**② 표 유형 분류 (wide vs key_value)**  
- `wide`: 행이 데이터이고 열이 속성인 일반 표 → Markdown으로 직렬화  
- `key_value`: 2열로 구성되고 첫 열이 짧은 레이블인 표(예: 사업명/사업금액) → `키: 값.` 문장 직렬화  
Markdown 표는 임베딩 시 구조가 일부 소실되지만, key_value는 자연어로 변환되어 벡터 검색 품질이 더 높음.

**③ 헤더 감지**  
한국 공공 RFP 문서는 `Ⅰ/Ⅱ/Ⅲ` (로마자), `1. 2. 3.` (숫자), `□` (체크박스), `가. 나. 다.` (한글) 패턴이 일관됨.  
정규식으로 3단계 계층 탐지. 줄 길이 80자 초과는 본문으로 판정 (헤더는 짧음).

**④ TableCell 내부 Paragraph 스킵**  
lxml의 `iter()`는 모든 후손을 순회하므로, 표 셀 내부 문단이 본문 문단으로 중복 수집됨.  
`iterancestors()`로 `TableCell`이 상위에 있으면 스킵.

---
## ⚠️ 패치 노트
## HWP 표 중복 추출 문제

### 발견된 근본 원인 :
표가 HWP 문단(Paragraph)에 '닻(anchor)'으로 걸려 있을 때, 기존 코드가 그
문단의 텍스트를 `el.itertext()`로 추출하면서 **표 셀 내부 텍스트까지 전부
같이 끌려와** "표 내용과 동일한 text 블록"이 표 바로 앞에 하나 더 생기고
있었습니다. 이는 장식용 표만의 문제가 아니라 **모든 표(데이터 표 포함)에서
구조적으로 발생하는 파서 버그**였습니다.

실제 한영대학교 D001 원본 HWP를 직접 XML로 떠서 검증한 결과:
- 수정 전: *표 110개 중 85개(77.3%)에서 직전 text 블록과 97% 이상 동일한 내용 중복 확인했습니다.*
- `paragraph_own_text()` 적용 후: 동일 조건 재검사 시 **중복 0건**으로 수정되었습니다

### 이번 버전에서 수정한 것
1. **(근본 수정)** `paragraph_own_text()` 함수 추가 — 문단 텍스트 추출 시 표/그림 등
   객체(Control) 내부를 건너뛰어 중복 발생 자체를 차단
2. **(버그 수정)** `is_decorative` / `decorative_reason` 필드가 표 블록 딕셔너리에
   실제로 기록되지 않던 문제 수정 (지난 버전은 합계만 세고 블록엔 기록 안 함)
3. **(임계값 보정)** `classify_decorative()`를 보조 안전장치로 유지하되, 임계값을
   보수적으로 조정(`text_overlap` 0.85→0.97, `tiny_table` 셀 2개→1개)하여
   진짜 데이터 표를 장식용으로 오판하지 않도록 함

---


In [ ]:
# ── HWP 파싱 유틸리티 ─────────────────────────────────────────

# 헤더 구조 설정
HEADER_PATTERNS = [
    (1, re.compile(r'^[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVXivx]+[\s\.·]')),
    (1, re.compile(r'^제\s*\d+\s*장\s')),
    (2, re.compile(r'^\d+\.\s+[가-힣A-Za-z]')),
    (2, re.compile(r'^□\s')),
    (3, re.compile(r'^\s{0,4}[가나다라마바사아자차카타파하]\.\s')),
    (3, re.compile(r'^\s{0,4}\d+\)\s')),
]

def detect_header_level(line: str) -> int:
    """문단 텍스트가 'Ⅰ. ', '1. ', '가. ' 같은 헤더 패턴인지 보고 레벨(1~3)을 반환. 아니면 0(본문)"""
    s = line.strip()
    if len(s) > 80: return 0  # 너무 긴 문장은 헤더가 아니라 본문으로 간주
    for lv, pat in HEADER_PATTERNS:
        if pat.match(s): return lv
    return 0


def cell_text(cell) -> str:
    """표 셀(TableCell) 하나의 텍스트를 추출. 셀 안은 itertext()로 긁어도 안전함
    (셀이 그 자체로 '진짜 표 내용'이므로 문단 추출 때와는 상황이 다름)"""
    return ' '.join(''.join(cell.itertext()).split())


# ──────────────────────────────────────────────────────────────
# [신규 추가 / 근본 원인 수정 함수]
# ──────────────────────────────────────────────────────────────
SKIP_CONTROL_TAGS = {
    'TableControl',         # 표
    'GShapeObjectControl',  # 도형/그림
    'EqEdit',               # 수식
    'ShapeComponent',       # 그림 구성요소
}

def paragraph_own_text(para_el) -> str:
    """
    문단(Paragraph) 요소에서 '그 문단 자체의 글자'만 추출한다.

    기존 코드의 el.itertext()는 문단 하위의 모든 요소(표, 그림 등)의
    텍스트까지 재귀적으로 다 끌어왔다. 이 함수는 SKIP_CONTROL_TAGS에 해당하는
    태그(표/그림/수식 등 '객체')를 만나면 그 하위로는 내려가지 않고 건너뛴다.

    이렇게 하면:
      - 표가 닻을 내린 '거의 빈' 문단은 빈 문자열을 반환 → body_items에
        text 블록으로 추가되지 않음 (= 표 내용이 중복 추출되지 않음)
      - 일반 문단은 기존과 동일하게 정상적으로 텍스트를 추출함
    """
    texts = []

    def walk(el):
        for child in el:
            if child.tag in SKIP_CONTROL_TAGS:
                # 표/그림/수식 등 '객체'를 만나면 그 내부는 재귀하지 않고 통째로 건너뜀
                continue
            if child.tag == 'Text' and child.text:
                texts.append(child.text)
            walk(child)

    walk(para_el)
    return ' '.join(''.join(texts).split())


def reconstruct_grid(table_el) -> list:
    """HWP TableBody → 2D 그리드 (병합 셀 복원)
    rowspan/colspan에 걸쳐 있는 모든 칸에 같은 텍스트를 채워 넣어
    표 모양 그대로의 2차원 배열을 만든다."""
    rows = int(table_el.get('rows', 1))
    cols = int(table_el.get('cols', 1))
    grid = [['' for _ in range(cols)] for _ in range(rows)]
    for cell in table_el.findall('.//TableCell'):
        r  = int(cell.get('row', 0))
        c  = int(cell.get('col', 0))
        rs = int(cell.get('rowspan', 1))
        cs = int(cell.get('colspan', 1))
        txt = cell_text(cell)
        for rr in range(r, min(r+rs, rows)):
            for cc in range(c, min(c+cs, cols)):
                grid[rr][cc] = txt
    return grid


def classify_table(grid: list) -> str:
    """표가 '세로형 2열 키-값 표'(예: 사업명: ...)인지, '가로형 일반 표'인지 구분"""
    if not grid: return 'wide'
    if len(grid[0]) == 2 and len(grid) >= 2:
        first_col = [r[0] for r in grid if r[0]]
        if first_col and sum(len(v) for v in first_col)/len(first_col) < 20:
            return 'key_value'
    return 'wide'


def grid_to_markdown(grid: list) -> str:
    """2D 그리드 → Markdown 표 문자열"""
    if not grid: return ''
    cols = len(grid[0])
    lines = ['| ' + ' | '.join(grid[0]) + ' |',
             '| ' + ' | '.join(['---']*cols) + ' |']
    for row in grid[1:]:
        lines.append('| ' + ' | '.join(row) + ' |')
    return '\n'.join(lines)


def grid_to_kv(grid: list) -> str:
    """2D 그리드(2열) → '키: 값.' 형태의 문장으로 직렬화"""
    return ' '.join(
        f"{r[0]}: {r[1]}."
        for r in grid if len(r) >= 2 and r[0]
    )


def serialize_table(grid: list) -> tuple:
    """그리드 → (table_type, content, note)"""
    t_type = classify_table(grid)
    if t_type == 'key_value':
        return t_type, grid_to_kv(grid), '세로형 2열 표 → 키-값 문장으로 직렬화'
    return t_type, grid_to_markdown(grid), '가로형 표 → Markdown으로 직렬화'


# ──────────────────────────────────────────────────────────────
# [보조 안전장치] 장식용 표(decorative table) 판별
#   - 근본 원인(위 paragraph_own_text)을 고쳤기 때문에, 이 함수가 잡는 케이스는
#     이제 매우 드물어야 정상입니다 (실제 검증 결과: 한영대학교 D001 표 110개 중 0건).
#   - 그래도 혹시 모를 디자인상의 진짜 중복(표지 제목을 텍스트로도 쓰고
#     박스 표로도 또 그린 경우 등)을 위한 안전망으로 유지합니다.
#   - 임계값을 보수적으로 조정한 이유:
#       · tiny_table: 셀 2개 이하 → 셀 1개(완전 단일 텍스트)로 좁힘
#         (셀 2개짜리 작은 표까지 장식용으로 단정하면 진짜 데이터 표를
#          오판할 위험이 있음)
#       · text_overlap: 0.85 → 0.97로 상향
#         (실제 데이터 표 중에도 직전 텍스트와 일부 겹치는 경우가 있어서,
#          '97% 동일'한 경우만 장식용으로 인정하도록 보수적으로 변경)
# ──────────────────────────────────────────────────────────────
NUMERIC_PATTERN = re.compile(r'^[\d,\.\-/%원₩()]+$')

def _norm(s: str) -> str:
    """비교를 위해 공백을 모두 제거한 정규화 문자열"""
    return re.sub(r'\s+', '', str(s))

def _table_plain_text(grid: list) -> str:
    """표 그리드의 모든 셀 텍스트를 이어붙인 문자열"""
    return ''.join(c for row in grid for c in row if c)

def _text_similarity(a: str, b: str) -> float:
    """두 문자열의 유사도(0~1). difflib 기반"""
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()

def _internal_repetition_ratio(grid: list) -> float:
    """표 내부에서 병합 셀 등으로 같은 텍스트가 여러 칸에 반복되는 비율
    (예: 목차 표에서 '목 차'라는 글자가 여러 칸에 반복되는 경우)"""
    cells = [_norm(c) for row in grid for c in row if c.strip()]
    if not cells:
        return 0.0
    return 1 - (len(set(cells)) / len(cells))

def _numeric_cell_ratio(grid: list) -> float:
    """셀 중 숫자/금액/날짜 패턴인 셀의 비율 (높을수록 '진짜 데이터 표'일 가능성↑)"""
    cells = [c.strip() for row in grid for c in row if c.strip()]
    if not cells:
        return 0.0
    return sum(1 for c in cells if NUMERIC_PATTERN.match(c)) / len(cells)

def classify_decorative(grid: list, prev_block: dict) -> tuple:
    """
    표 grid와 '바로 직전 블록'을 보고 (장식용 여부: bool, 판정 이유: str)를 반환한다.
    """
    n_rows = len(grid)
    n_cols = len(grid[0]) if grid else 0
    nonempty = [c for row in grid for c in row if c.strip()]

    # 신호1: 표가 완전히 '문장 한 줄'짜리(1행 1열)인 경우 → 표지 제목 박스 등
    if n_rows == 1 and n_cols == 1:
        return True, f'tiny_table(rows={n_rows},cols={n_cols})'

    # 신호2: 표 전체 텍스트가 바로 위 텍스트 블록과 사실상 동일(97% 이상)
    plain = _norm(_table_plain_text(grid))
    sim = 0.0
    if prev_block is not None and prev_block.get('type') == 'text':
        sim = _text_similarity(plain, _norm(prev_block['content']))
        if sim >= 0.97:
            return True, f'text_overlap={sim:.2f}'

    # 신호3: 병합 셀로 인한 내부 반복이 심하고 숫자 데이터가 거의 없음
    rep = _internal_repetition_ratio(grid)
    numr = _numeric_cell_ratio(grid)
    if rep >= 0.5 and numr < 0.1:
        return True, f'internal_repetition={rep:.2f}'

    return False, f'data_table(sim={sim:.2f},rep={rep:.2f},numeric={numr:.2f})'


# D026, D043 파싱 실패 후 키에러 나서 맞추기 위한 코드 ────────────────
def _empty_qa_info(warning_msg: str) -> dict:
    """파싱 실패 시에도 모든 키를 가진 qa_info를 반환 (다운스트림 KeyError 방지)"""
    return {
        'total_sections': 0, 'total_blocks': 0,
        'text_blocks': 0, 'table_blocks': 0,
        'table_wide_count': 0, 'table_key_value_count': 0,
        'decorative_table_count': 0, 'decorative_table_ratio': 0.0,
        'extraction_warnings': [warning_msg]
    }


def _flush_section(sections, sec_counter, current_headers, current_level, current_blocks):
    """섹션 확정 시 block_id를 'S01-B01' 형식(섹션 prefix)으로 재부여"""
    if current_blocks:
        sec_counter += 1
        section_id = f'S{str(sec_counter).zfill(2)}'
        renumbered_blocks = []
        for i, blk in enumerate(current_blocks, start=1):
            blk = dict(blk)
            blk['block_id'] = f'{section_id}-B{str(i).zfill(2)}'
            renumbered_blocks.append(blk)
        sections.append({
            'section_id':  section_id,
            'header_path': list(current_headers),
            'level':       current_level,
            'blocks':      renumbered_blocks
        })
    return sec_counter


def parse_hwp(hwp_path: Path) -> tuple:
    """
    HWP 파일 → (sections, qa_info)
    sections: 헤더 기준으로 분할된 섹션 배열
    qa_info : 전처리 품질 정보
    """
    result = subprocess.run(
        ['hwp5proc', 'xml', str(hwp_path)], capture_output=True
    )
    if result.returncode != 0:
        return [], _empty_qa_info(f'hwp5proc 실패: {hwp_path.name}')

    with tempfile.NamedTemporaryFile(suffix='.xml', delete=False) as f:
        f.write(result.stdout)
        tmp = f.name
    try:
        tree = etree.parse(tmp)
    except Exception as e:
        os.unlink(tmp)
        return [], _empty_qa_info(f'XML 파싱 실패: {e}')
    os.unlink(tmp)

    root = tree.getroot()

    # 문서 순서대로 문단·표 수집
    body_items = []
    for el in root.iter():
        if el.tag == 'TableBody':
            body_items.append(('table', el))
        elif el.tag == 'Paragraph':
            parent_tags = {p.tag for p in el.iterancestors()}
            if 'TableCell' not in parent_tags:
                # ★★★ 근본 원인 수정 지점 ★★★
                # 기존: txt = ' '.join(''.join(el.itertext()).split())
                #   → 문단 하위에 표가 닻을 내리고 있으면 표 셀 텍스트까지
                #      전부 같이 긁혀서 '표 내용과 동일한 text 블록'이 생겼음
                # 수정: paragraph_own_text() 사용
                #   → 표/그림 등 객체(Control) 내부는 건너뛰고 문단 자체
                #      글자만 추출 → 표가 닻을 내린 문단은 빈 문자열이 되어
                #      자동으로 text 블록 후보에서 제외됨
                txt = paragraph_own_text(el)
                if txt:
                    body_items.append(('text', txt))

    sections, current_headers, current_level, current_blocks = [], ['(서두)'], 0, []
    sec_counter = blk_counter = text_count = table_count = 0
    wide_count = kv_count = decorative_count = 0
    warnings = []

    for itype, ival in body_items:
        if itype == 'text':
            lv = detect_header_level(ival)
            if lv > 0:
                sec_counter = _flush_section(sections, sec_counter, current_headers, current_level, current_blocks)
                current_blocks = []
                if lv == 1:   current_headers = [ival.strip()]
                elif lv == 2: current_headers = (current_headers[:1] + [ival.strip()])
                else:         current_headers = (current_headers[:2] + [ival.strip()])
                current_level = lv
            else:
                blk_counter += 1; text_count += 1
                current_blocks.append({'block_id': f'B{str(blk_counter).zfill(4)}',
                                       'type': 'text', 'content': ival})
        else:
            grid = reconstruct_grid(ival)
            if not grid or not any(any(c for c in r) for r in grid):
                warnings.append(f'빈 표 감지 (blk#{blk_counter+1})')
                continue
            t_type, content, note = serialize_table(grid)
            if not content.strip():
                warnings.append(f'표 직렬화 빈 결과 (blk#{blk_counter+1})')
                continue
            blk_counter += 1; table_count += 1
            wide_count += 1 if t_type == 'wide' else 0
            kv_count   += 1 if t_type == 'key_value' else 0

            # 장식용 표 판별 (보조 안전장치) — 직전 블록과 비교
            prev_block = current_blocks[-1] if current_blocks else None
            is_dec, dec_reason = classify_decorative(grid, prev_block)
            if is_dec:
                decorative_count += 1

            # ★★★ 지난번 누락됐던 부분(버그) 수정 ★★★
            # is_decorative / decorative_reason을 실제 블록 딕셔너리에
            # 반드시 기록해야 청킹/임베딩 단계에서 block.get('is_decorative')로
            # 걸러낼 수 있다. (이전 버전은 qa 합계만 세고 블록엔 안 적었음)
            current_blocks.append({'block_id': f'B{str(blk_counter).zfill(4)}',
                                   'type': 'table', 'table_type': t_type,
                                   'content': content, 'note': note, 'raw_grid': grid,
                                   'is_decorative': is_dec,
                                   'decorative_reason': dec_reason})

    _flush_section(sections, sec_counter, current_headers, current_level, current_blocks)

    return sections, {
        'total_sections': len(sections), 'total_blocks': blk_counter,
        'text_blocks': text_count, 'table_blocks': table_count,
        'table_wide_count': wide_count, 'table_key_value_count': kv_count,
        'decorative_table_count': decorative_count,
        'decorative_table_ratio': round(decorative_count/table_count, 3) if table_count else 0.0,
        'extraction_warnings': warnings
    }

print('HWP 파싱 유틸리티 정의 완료 (근본 원인 수정 + decorative 안전장치 반영)')


**결과**

HWP 파싱 유틸리티 정의 완료 (근본 원인 수정 + decorative 안전장치 반영)

---
##4.PDF 파싱 유틸리티

### 처리 흐름
```
PDF 파일
  └─▶ pdfplumber.open()
        ├─▶ page.find_tables()    → 표 bbox 목록
        ├─▶ page.extract_text()   → 텍스트 (표 영역 제외)
        └─▶ page.extract_tables() → 표 셀 배열
```

### 핵심 설계 결정

**① 표 영역 텍스트 분리**  
pdfplumber의 `extract_text()`는 표 안의 텍스트도 함께 추출함.  
표 bbox를 먼저 구한 뒤 해당 영역 객체를 `filter()`로 제외하여  
본문 텍스트와 표 텍스트가 중복 수집되는 것을 방지.

**② 병합 셀(None) 처리**  
pdfplumber는 병합 셀의 빈 자리를 `None`으로 반환.  
첫 번째 열의 `None`은 위 행 값으로 채워 병합 셀을 근사 복원.  
(HWP처럼 정확한 rowspan 메타가 없으므로 근사 처리)

**③ HWP와 동일한 스키마 출력**  
청킹팀이 파일 포맷을 구분하지 않아도 되도록,  
PDF 파싱 결과도 HWP와 완전히 동일한 `sections/blocks` 구조로 반환.

In [ ]:
# ── PDF 파싱 유틸리티 ─────────────────────────────────────────
#
# [참고] PDF는 HWP와 달리 pdfplumber가 처음부터 표 영역(bbox)을 따로 잡아서
# "표 영역을 제외한 텍스트"만 본문으로 추출하므로(아래 filtered.extract_text()),
# HWP에서 발견된 '문단이 표를 통째로 끌어안는' 근본 원인 버그는 구조적으로
# 발생하지 않습니다.
# ────────────────────────────────────────────────────────────────

def pdf_clean_grid(raw_table: list) -> list:
    """pdfplumber 표 → None 처리 + 공백 정규화 + 빈 행 제거"""
    cleaned, prev_first = [], ''
    for row in raw_table:
        new_row = []
        for i, cell in enumerate(row):
            val = str(cell).strip() if cell is not None else ''
            if i == 0:
                val = val if val else prev_first   # 병합 셀 근사 복원
                prev_first = val
            val = re.sub(r'\s+', ' ', val.replace('\n', ' ').replace('\xad', '-'))
            new_row.append(val)
        if any(c for c in new_row):
            cleaned.append(new_row)
    return cleaned

def parse_pdf(pdf_path: Path) -> tuple:
    """
    PDF 파일 → (sections, qa_info)  ← HWP와 동일한 스키마
    """
    sections, current_headers, current_level, current_blocks = [], ['(서두)'], 0, []
    sec_counter = blk_counter = text_count = table_count = 0
    wide_count = kv_count = decorative_count = 0
    warnings = []

    try:
        with pdfplumber.open(str(pdf_path)) as pdf:
            for page in pdf.pages:
                page_tables = page.extract_tables()
                try:
                    table_bboxes = [t.bbox for t in page.find_tables()]
                except Exception:
                    table_bboxes = []

                # 표 영역 제외 후 텍스트 추출
                if table_bboxes:
                    filtered = page
                    for bbox in table_bboxes:
                        filtered = filtered.filter(
                            lambda obj, b=bbox: not (
                                obj.get('x0',0) >= b[0] and obj.get('x1',0) <= b[2] and
                                obj.get('top',0) >= b[1] and obj.get('bottom',0) <= b[3]
                            )
                        )
                    raw_text = filtered.extract_text() or ''
                else:
                    raw_text = page.extract_text() or ''

                # 텍스트 처리
                for line in raw_text.split('\n'):
                    line = line.strip()
                    if not line: continue
                    lv = detect_header_level(line)
                    if lv > 0:
                        sec_counter = _flush_section(sections, sec_counter, current_headers, current_level, current_blocks)
                        current_blocks = []
                        if lv == 1:   current_headers = [line]
                        elif lv == 2: current_headers = (current_headers[:1] + [line])
                        else:         current_headers = (current_headers[:2] + [line])
                        current_level = lv
                    else:
                        blk_counter += 1; text_count += 1
                        current_blocks.append({'block_id': f'B{str(blk_counter).zfill(4)}',
                                               'type': 'text', 'content': line})

                # 표 처리
                for raw_table in page_tables:
                    grid = pdf_clean_grid(raw_table)
                    if not grid: continue
                    t_type, content, note = serialize_table(grid)
                    if not content.strip(): continue
                    blk_counter += 1; table_count += 1
                    wide_count += 1 if t_type == 'wide' else 0
                    kv_count   += 1 if t_type == 'key_value' else 0

                    # 장식용 표 판별 (보조 안전장치)
                    prev_block = current_blocks[-1] if current_blocks else None
                    is_dec, dec_reason = classify_decorative(grid, prev_block)
                    if is_dec:
                        decorative_count += 1

                    # ★★★ 지난번 누락됐던 부분(버그) 수정 ★★★
                    # HWP와 동일하게, is_decorative / decorative_reason을
                    # 실제 블록 딕셔너리에 기록한다.
                    current_blocks.append({'block_id': f'B{str(blk_counter).zfill(4)}',
                                           'type': 'table', 'table_type': t_type,
                                           'content': content, 'note': note, 'raw_grid': grid,
                                           'is_decorative': is_dec,
                                           'decorative_reason': dec_reason})

    except Exception as e:
        warnings.append(f'PDF 파싱 오류: {e}')

    _flush_section(sections, sec_counter, current_headers, current_level, current_blocks)

    return sections, {
        'total_sections': len(sections), 'total_blocks': blk_counter,
        'text_blocks': text_count, 'table_blocks': table_count,
        'table_wide_count': wide_count, 'table_key_value_count': kv_count,
        'decorative_table_count': decorative_count,
        'decorative_table_ratio': round(decorative_count/table_count, 3) if table_count else 0.0,
        'extraction_warnings': warnings
    }

print('PDF 파싱 유틸리티 정의 완료 (필드 누락 버그 수정 반영)')


**결과**

PDF 파싱 유틸리티 정의 완료 (필드 누락 버그 수정 반영)

---
## 5.JSON 빌드 함수 정의

### 결측치 `<unknown>` 적용

CSV에서 이미 `<unknown>`으로 치환된 값이 JSON `metadata` 필드에 그대로 들어갑니다.  
청킹 시 `metadata.입찰참여시작일 == '<unknown>'` 조건으로 결측 여부를 판단할 수 있습니다.

### qa 필드 보강
- `table_wide_count` / `table_key_value_count`: 표 직렬화 방식별 개수
- `toc_header_match_rate`: 추출된 목차 항목이 실제 헤더 탐지 결과에 등장하는 비율 (헤더 탐지 정확도 지표)
- `dedup_hash`: 전체 본문 텍스트 SHA-256 해시 (중복 문서 확인용)

In [ ]:
TOC_PATTERN = re.compile(
    r'^([ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVXivx]+[^\n]{2,40}?|\d+\.\s[가-힣A-Za-z][^\n]{2,40}?)'
    r'[\s·\-·⋯\.]{3,}\s*\d+',
    re.MULTILINE
)

def extract_toc(text: str) -> list:
    return [m.strip() for m in TOC_PATTERN.findall(str(text))][:30]

def compute_toc_match_rate(toc: list, sections: list) -> float:
    """toc 항목이 실제 header_path에 등장하는 비율 (자동 헤더 탐지 검증용)"""
    if not toc:
        return 0.0
    all_headers = ' '.join(
        h for s in sections for h in s['header_path']
    )
    matched = sum(1 for t in toc if t[:6] in all_headers)
    return round(matched / len(toc), 2)

def build_json(row: pd.Series, sections: list, qa_info: dict, toc: list) -> dict:
    all_text = ' '.join(b['content'] for s in sections for b in s['blocks'])
    qa_info['toc_header_match_rate'] = compute_toc_match_rate(toc, sections)
    qa_info['dedup_hash'] = 'sha256:' + hashlib.sha256(all_text.encode()).hexdigest()

    # ── 추가: 장식용 표 비율이 높으면 자동 경고 ──────────────────
    dec_ratio = qa_info.get('decorative_table_ratio', 0.0)
    if dec_ratio > 0.15:
        qa_info['extraction_warnings'].append(
            f'high_decorative_table_ratio: {dec_ratio:.1%}'
        )

    return {
        'schema_version': '1.0',
        'doc_id':         row['doc_id'],
        'file_name':      str(row['파일명']),
        'source_format':  str(row['파일형식']).lower(),
        'processed_at':   str(date.today()),
        'metadata': {
            '공고번호':       row['공고번호'],        # 결측 시 <unknown>
            '공고차수':       int(row['공고차수']),
            '사업명':         row['사업명'],
            '사업금액':       int(row['사업금액']) if pd.notna(row['사업금액']) else None,
            '발주기관':       row['발주기관'],
            '공개일자':       row['공개일자'],        # 결측 시 <unknown>
            '입찰참여시작일': row['입찰참여시작일'],  # 결측 시 <unknown>
            '입찰참여마감일': row['입찰참여마감일'],  # 결측 시 <unknown>
            '사업요약':       row['사업요약'],        # 결측 시 <unknown>
            '사업유형':       row['사업유형'],
            '재공고여부':     bool(row['재공고여부']),
            'linked_doc_id':  None,
            '목차존재':       len(toc) > 0
        },
        'toc':      toc,
        'sections': sections,
        'qa':       qa_info
    }

print('JSON 빌드 함수 정의 완료')

**결과**

JSON 빌드 함수 정의 완료

---
##6.파싱 예시 출력

본 파이프라인을 실제 파일에 적용했을 때 JSON이 어떻게 생성되는지 확인합니다.  
전체 실행 전 결과물의 구조와 품질을 미리 점검하는 목적입니다.

예시 JSON은 `build_json()`을 그대로 사용하므로 실제 100건 산출물과 완전히 동일한 스키마/형식입니다.

In [ ]:
# ── HWP 파싱 예시 (1건) ───────────────────────────────────────
# 첫 번째 HWP 파일에 적용하여 JSON 구조 미리보기
# build_json()을 그대로 재사용하므로 실제 100건 산출물과 완전히 동일한 스키마/형식

sample_hwp = hwp_files[0]
sample_row = df[df['file_path'].apply(lambda p: p == sample_hwp if p else False)]
if sample_row.empty:
    sample_row = df[df['파일형식'].str.lower() == 'hwp'].iloc[0:1]
    sample_hwp = sample_row.iloc[0]['file_path']

print(f'대상 파일: {sample_hwp.name}')
sections_hwp, qa_hwp = parse_hwp(sample_hwp)
row = sample_row.iloc[0]
toc_hwp = extract_toc(row.get('텍스트', ''))

hwp_full_json = build_json(row, sections_hwp, qa_hwp, toc_hwp)

# 표 포함 섹션 2개 + 텍스트 섹션 1개만 골라서 출력 (전체는 너무 길어서 일부만 미리보기)
table_secs = [s for s in hwp_full_json['sections'] if any(b['type']=='table' for b in s['blocks'])][:2]
text_secs  = [s for s in hwp_full_json['sections'] if all(b['type']=='text'  for b in s['blocks'])][:1]
hwp_example_json = dict(hwp_full_json)
hwp_example_json['sections'] = (table_secs + text_secs)[:3]

print('\n=== HWP 파싱 예시 JSON (섹션 샘플, 실제 산출물과 동일한 스키마) ===')
print(json.dumps(hwp_example_json, ensure_ascii=False, indent=2))

In [ ]:
# ── PDF 파싱 예시 (1건) ───────────────────────────────────────
# build_json()을 그대로 재사용하므로 실제 100건 산출물과 완전히 동일한 스키마/형식

sample_pdf = pdf_files[0]
sample_pdf_row = df[df['파일형식'].str.lower() == 'pdf'].iloc[0]

print(f'대상 파일: {sample_pdf.name}')
sections_pdf, qa_pdf = parse_pdf(sample_pdf)
toc_pdf = extract_toc(sample_pdf_row.get('텍스트', ''))

pdf_full_json = build_json(sample_pdf_row, sections_pdf, qa_pdf, toc_pdf)

table_secs_pdf = [s for s in pdf_full_json['sections'] if any(b['type']=='table' for b in s['blocks'])][:2]
text_secs_pdf  = [s for s in pdf_full_json['sections'] if all(b['type']=='text'  for b in s['blocks'])][:1]
pdf_example_json = dict(pdf_full_json)
pdf_example_json['sections'] = (table_secs_pdf + text_secs_pdf)[:3]

print('\n=== PDF 파싱 예시 JSON (섹션 샘플) ===')
print(json.dumps(pdf_example_json, ensure_ascii=False, indent=2))

---
##7.전체 파이프라인 실행

In [ ]:
# ── 전체 실행 ─────────────────────────────────────────────────
manifest_rows = []
failed_docs   = []

for idx, row in df.iterrows():
    doc_id    = row['doc_id']
    file_path = row['file_path']
    fmt       = str(row['파일형식']).lower()

    print(f'[{doc_id}] {str(row["사업명"])[:35]}...', end=' ')

    if file_path is None or not Path(str(file_path)).exists():
        print('❌ 파일 없음')
        failed_docs.append({'doc_id': doc_id, '사업명': row['사업명'], '원인': '파일 없음'})
        continue

    try:
        if fmt == 'hwp':
            sections, qa_info = parse_hwp(Path(str(file_path)))
        elif fmt == 'pdf':
            sections, qa_info = parse_pdf(Path(str(file_path)))
        else:
            print(f'❌ 미지원 형식({fmt})')
            failed_docs.append({'doc_id': doc_id, '사업명': row['사업명'], '원인': f'미지원 형식 {fmt}'})
            continue
    except Exception as e:
        print(f'❌ 파싱 예외: {e}')
        failed_docs.append({'doc_id': doc_id, '사업명': row['사업명'], '원인': str(e)})
        continue

    # 파싱 결과가 완전히 비어있으면 실패로 분류 (JSON 저장 안 함) ──
    if qa_info['total_sections'] == 0:
        reason = qa_info['extraction_warnings'][0] if qa_info['extraction_warnings'] else '알 수 없는 파싱 실패'
        print(f'❌ 파싱 결과 없음: {reason}')
        failed_docs.append({'doc_id': doc_id, '사업명': row['사업명'], '원인': reason})
        continue

    toc = extract_toc(row.get('텍스트', ''))
    doc_json = build_json(row, sections, qa_info, toc)

    out_path = DOCS_DIR / f'{doc_id}.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(doc_json, f, ensure_ascii=False, indent=2)

    has_warn = len(qa_info['extraction_warnings']) > 0
    print(f'✅ 섹션={qa_info["total_sections"]} 블록={qa_info["total_blocks"]} '
          f'표={qa_info["table_blocks"]} {"⚠️" if has_warn else ""}')

    manifest_rows.append({
        'doc_id': doc_id, '파일명': row['파일명'], '사업명': row['사업명'],
        '발주기관': row['발주기관'], '사업유형': row['사업유형'],
        '사업금액': row['사업금액'], '공개일자': row['공개일자'],
        '파일형식': fmt, '재공고여부': row['재공고여부'],
        '목차존재': len(toc) > 0,
        'total_sections': qa_info['total_sections'],
        'total_blocks':   qa_info['total_blocks'],
        'table_blocks':   qa_info['table_blocks'],
        'has_warning':    has_warn,
        'json_path':      str(out_path)
    })

print(f'\n완료: {len(manifest_rows)}건 성공 / {len(failed_docs)}건 실패')

---
##8.manifest 저장 & 재공고 연결

In [ ]:
# ── 재공고 linked_doc_id 연결 ─────────────────────────────────
# 동일 사업명 + 발주기관에서 공고차수가 낮은 것이 원공고

manifest_df = pd.DataFrame(manifest_rows)

for _, re_row in df[df['재공고여부'] == True].iterrows():
    candidates = df[
        (df['사업명'] == re_row['사업명']) &
        (df['발주기관'] == re_row['발주기관']) &
        (df['공고차수'] < re_row['공고차수'])
    ]
    if candidates.empty: continue
    original_id = candidates.sort_values('공고차수').iloc[0]['doc_id']

    json_path = DOCS_DIR / f"{re_row['doc_id']}.json"
    if json_path.exists():
        with open(json_path, 'r', encoding='utf-8') as f:
            doc = json.load(f)
        doc['metadata']['linked_doc_id'] = original_id
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(doc, f, ensure_ascii=False, indent=2)
        print(f"재공고 연결: {re_row['doc_id']} → 원공고 {original_id}")

print('재공고 연결 완료')

In [ ]:
# ── manifest.csv 저장 ─────────────────────────────────────────
manifest_path = OUTPUT_DIR / 'manifest.csv'
manifest_df.to_csv(manifest_path, index=False, encoding='utf-8-sig')
print(f'manifest 저장: {manifest_path}')

if failed_docs:
    fail_df = pd.DataFrame(failed_docs)
    fail_path = OUTPUT_DIR / 'failed_docs.csv'
    fail_df.to_csv(fail_path, index=False, encoding='utf-8-sig')
    print(f'실패 목록 저장: {fail_path}')
    print(fail_df)

print('\n=== 최종 요약 ===')
print(f'성공:           {len(manifest_rows)}건')
print(f'실패:           {len(failed_docs)}건')
print(f'경고 있는 문서: {manifest_df["has_warning"].sum()}건')
print(f'표 없는 문서:   {(manifest_df["table_blocks"] == 0).sum()}건  ← 파싱 실패 의심')
print(f'출력 경로:      {OUTPUT_DIR}')
manifest_df.head()

---
## QA. 샘플 검증

특정 문서의 JSON을 열어 섹션·표 구조를 확인합니다.  
`CHECK_DOC_ID` 를 바꿔가며 여러 문서를 점검할 수 있습니다

`CHECK_DOC_ID`는 현재 D001 ~ D099까지 있으며 D026과 D043은 파싱 실패로 없는 문서입니다

In [ ]:
CHECK_DOC_ID = 'D001'   # ← 확인할 doc_id 변경

with open(DOCS_DIR / f'{CHECK_DOC_ID}.json', 'r', encoding='utf-8') as f:
    doc = json.load(f)

m = doc['metadata']
print(f"사업명         : {m['사업명']}")
print(f"발주기관       : {m['발주기관']}")
print(f"사업유형       : {m['사업유형']}")
print(f"입찰참여시작일 : {m['입찰참여시작일']}  (결측이면 <unknown>)")
print(f"섹션 수        : {doc['qa']['total_sections']}")
print(f"블록 수        : {doc['qa']['total_blocks']}  (표: {doc['qa']['table_blocks']})")
print(f"경고           : {doc['qa']['extraction_warnings']}")
print()

for sec in doc['sections'][:3]:
    print(f"[{sec['section_id']}] {' > '.join(sec['header_path'])}")
    for blk in sec['blocks'][:2]:
        preview = blk['content'][:80].replace('\n', ' ')
        print(f"  [{blk['type']}] {preview}...")
    print()

In [ ]:
# 경고 문서 & 표 없는 문서 목록
warn_docs     = manifest_df[manifest_df['has_warning']][['doc_id','사업명','table_blocks']]
no_table_docs = manifest_df[manifest_df['table_blocks']==0][['doc_id','사업명']]

print(f'⚠️  경고 문서 {len(warn_docs)}건:')
print(warn_docs.to_string(index=False))
print(f'\n🔴 표 없는 문서 {len(no_table_docs)}건 (파싱 실패 의심, 수동 확인 필요):')
print(no_table_docs.to_string(index=False))